In [11]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [12]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
WB_KEY = user_secrets.get_secret("wandb-key")


In [13]:
import wandb 
wandb.login(key=WB_KEY)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [14]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F

class ScratchMCQSolver(nn.Module): 
    def __init__(self, input_dim, hidden_dim=128): 
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 5)

    def forward(self, x): 
        x = F.relu(self.fc1(x))
        logits = self.fc2(x)
        return logits 
vocab_size = 5000 
model_scratch = ScratchMCQSolver(input_dim=vocab_size)
print(model_scratch)

ScratchMCQSolver(
  (fc1): Linear(in_features=5000, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=5, bias=True)
)


In [15]:
import pandas as pd 
import numpy as np 
import re 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity 

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [16]:
train_df.head()

,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [17]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
dtypes: int64(1), object(7)
memory usage: 125.1+ KB


In [18]:
train_df.fillna("None", inplace=True)

In [19]:
def clean_text(text): 
    text = str(text).lower()
    text = re.sub(r'\s{2,}', '', text)
    return text.strip()

columns_to_clean = ['prompt', 'A', 'B', 'C', 'D', 'E']
for col in columns_to_clean: 
    if col in train_df.columns: 
        train_df[col] = train_df[col].apply(clean_text)

print("Data Cleaned")

Data Cleaned


In [20]:
vectorizer = TfidfVectorizer(stop_words='english')
predictions = []
actuals = []

for idx, row in train_df.iterrows():
    corpus = [row['prompt'], row['A'], row['B'], row['C'], row['D'], row['E']]

    tfidf_matrix = vectorizer.fit_transform(corpus)
    prompt_vector = tfidf_matrix[0:1]
    options_matrix = tfidf_matrix[1:]

    similarities = cosine_similarity(prompt_vector, options_matrix).flatten()

    labels = ['A', 'B', 'C', 'D', 'E']
    top_3_idx = similarities.argsort()[-3:][::-1]
    top_3_preds = [labels[i] for i in top_3_idx]

    predictions.append(top_3_preds)
    if 'answer' in train_df.columns: 
        actuals.append(row['answer'])


In [21]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual)+1)

    return 0

if actuals: 
    map_3_score = np.mean([apk(a, p) for a, p in zip(actuals, predictions)])
    print(f'Baseline TF-IDF MAP@3: {map_3_score: .4f}')
else: 
    print('No answer column found')

Baseline TF-IDF MAP@3:  0.3260


# Mile 2

In [22]:
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [23]:
df_pandas=pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')
dataset = Dataset.from_pandas(df_pandas)

In [24]:
def combine_text_fn(example): 
    return {"combined_text": f"{example['prompt']} {example['A']}"}

dataset = dataset.map(combine_text_fn)

tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

def tokenize_prompts(examples): 
    return tokenizer(examples['prompt'], padding='max_length', truncation=True, max_length=128)

tokenized_dataset = dataset.map(tokenize_prompts, batched=True)
input_ids_shape = np.array(tokenized_dataset['input_ids']).shape

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [25]:
model = AutoModel.from_pretrained('bert-base-uncased')
row_0_prompt = dataset[0]['prompt']
inputs_0 = tokenizer(row_0_prompt, return_tensors='pt')

with torch.no_grad():
    outputs_0 = model(**inputs_0)

last_hidden_state = outputs_0.last_hidden_state 

cls_vector = last_hidden_state[0, 0, :5].tolist()

model_att = AutoModel.from_pretrained('bert-base-uncased', output_attentions=True)
text_att = 'Light-ion fusion is a technique.'
inputs_att = tokenizer(text_att, return_tensors='pt')
tokens_att = tokenizer.convert_ids_to_tokens(inputs_att['input_ids'][0])

fusion_idx = tokens_att.index('fusion')

with torch.no_grad(): 
    outputs_att = model_att(**inputs_att)

attention_matrix = outputs_att.attentions[-1][0, 0]
weight_cls_to_fusion = attention_matrix[0, fusion_idx].item()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [26]:
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
emb_prompt = st_model.encode(dataset[0]['prompt'], convert_to_tensor=True)
emb_opt_b = st_model.encode(dataset[0]['B'], convert_to_tensor=True)
sim_score = util.cos_sim(emb_prompt, emb_opt_b).item()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [27]:
def apk(actual, predicted, k=3): 
    predicted = predicted[:k]
    if actual in predicted: 
        return 1 / (predicted.index(actual) + 1)
    return 0 
labels = ['A', 'B', 'C', 'D', 'E']
tf_idf_vectorizer = TfidfVectorizer(stop_words='english')

minilm_preds = []
tfidf_preds = []
actual_answers = df_pandas['answer'].tolist() if 'answer' in df_pandas.columns else []

all_prompts = df_pandas['prompt'].tolist()
opt_cols = [df_pandas[c].tolist() for c in labels]

prompt_embs = st_model.encode(all_prompts, convert_to_tensor=True)
opt_embs = [st_model.encode(col, convert_to_tensor=True) for col in opt_cols]

for idx, row in df_pandas.iterrows(): 
    corpus = [row['prompt']] + [row[l] for l in labels]
    try: 
        tfidf_mat = tf_idf_vectorizer.fit_transform(corpus)
        sims_tfidf = cosine_similarity(tfidf_mat[0:1], tfidf_mat[1:]).flatten()
        top_3_tfidf = [labels[i] for i in sims_tfidf.argsort()[-3:][::-1]]

    except: 
        top_3_tfidf = ['A', 'B', 'C']
    tfidf_preds.append(top_3_tfidf)

    p_emb = prompt_embs[idx]
    sims_minilm = []
    for o_idx in range(5): 
        sims_minilm.append(util.cos_sim(p_emb, opt_embs[o_idx][idx]).item())

    top_3_minilm = [labels[i] for i in np.argsort(sims_minilm)[-3:][::-1]]
    minilm_preds.append(top_3_minilm)

map3_minilm = np.mean([apk(a, p) for a, p in zip(actual_answers, minilm_preds)])

improvement_count = 0
for a, t_pred, m_pred in zip(actual_answers, tfidf_preds, minilm_preds): 
    if(a in m_pred) and (a not in t_pred): 
        improvement_count += 1

In [28]:
classifier = pipeline('zero-shot-classification', model = 'facebook/bart-large-mnli', device=0 if torch.cuda.is_available() else -1)
prompt_idx_1 = dataset[1]['prompt']
candidates = [dataset[1]['A'], dataset[1]['B'], dataset[1]['C']]

res_softmax = classifier(prompt_idx_1, candidate_labels = candidates, multi_label=False)
top_prob_softmax = res_softmax['scores'][0]

res_sigmoid = classifier(prompt_idx_1, candidate_labels=candidates, multi_label=True)
abs_diff = abs(sum(res_softmax['scores']) - sum(res_sigmoid['scores']))

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [29]:
from transformers import AutoModelForSeq2SeqLM

In [30]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-small")
slm_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-small").to("cuda:0")
prompt_slm = f"Question: {dataset[0]['prompt']}. Is the correct answer A: {dataset[0]['A']} or B: {dataset[0]['B']}? Answer with just the letter A or B."
inputs = tokenizer(prompt_slm, return_tensors="pt").to("cuda:0")
outputs = slm_model.generate(**inputs, max_new_tokens=5)
slm_out = tokenizer.decode(outputs[0], skip_special_tokens=True)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

## Answers Milstone 2

In [31]:
print(f"Q1: Character length at index 51: {len(dataset[51]['combined_text'])}")
print(f"Q2: Vocabulary Size: {tokenizer.vocab_size}")
print(f"Q3: [SEP] Token ID: {tokenizer.convert_tokens_to_ids('[SEP]')}")
print(f"Q4: Geometric shape of input_ids tensor: {input_ids_shape}")
print(f"Q5: Dimensionality of each individual attention head: {768 // 12}")
print(f"Q6: Shape of last_hidden_state tensor: {list(last_hidden_state.shape)}")
print(f"Q7: Sum of first 5 float values in [CLS] vector: {round(sum(cls_vector), 4)}")
print(f"Q8: Attention weight from [CLS] to 'fusion': {round(weight_cls_to_fusion, 4)}")
print(f"Q9: Cosine similarity between prompt and Option B: {round(sim_score, 4)}")
print(f"Q10 Part 1: Final MAP@3 score of MiniLM pipeline: {round(map3_minilm, 4)}")
print(f"Q10 Part 2: Number of questions saved by MiniLM: {improvement_count}")

Q1: Character length at index 51: 614
Q2: Vocabulary Size: 32100
Q3: [SEP] Token ID: 2
Q4: Geometric shape of input_ids tensor: (2000, 128)
Q5: Dimensionality of each individual attention head: 64
Q6: Shape of last_hidden_state tensor: [1, 31, 768]
Q7: Sum of first 5 float values in [CLS] vector: -1.2001
Q8: Attention weight from [CLS] to 'fusion': 0.1025
Q9: Cosine similarity between prompt and Option B: 0.7658
Q10 Part 1: Final MAP@3 score of MiniLM pipeline: 0.4231
Q10 Part 2: Number of questions saved by MiniLM: 488


# Milestone 3

In [32]:
!pip install -q langchain-text-splitters langchain-huggingface faiss-cpu 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.6/561.6 kB 32.5 MB/s eta 0:00:00


In [33]:
!pip install -q langchain-community 

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 27.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.1 which is incompatible.
ydata-profiling 4.18.4 requires numpy<2.4,>=1.22, but you have numpy 2.4.6 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible

In [34]:
import os 
import re 
import numpy as np
import pandas as pd 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_core.documents import Document 
from langchain_huggingface import HuggingFaceEmbeddings 
from langchain_community.vectorstores import FAISS 

/tmp/ipykernel_59/3046257206.py:8: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


In [35]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')

In [36]:
external_knowledge_corpus = """
Light-ion fusion is a nuclear reaction technique where light atomic nuclei combine to form heavier elements.
Tragedy of Summerhall was a catastrophic fire event in 259 AC that led to the deaths of King Aegon V.
Aegon the Conqueror forged the Iron Throne using the fiery breath of his dragon, Balerion the Black Dread.
Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings.
Transformers rely on multi-head self-attention mechanisms to map contextual text representations dynamically.
"""
def clean_corpus_text(text):
    text = text.strip()
    text = re.sub(r' {2,}', ' ', text)
    text = text.replace("\r", "")
    return text

cleaned_corpus = clean_corpus_text(external_knowledge_corpus)

In [37]:
docs = [Document(page_content=cleaned_corpus, metadata={"source": "external_knowledge_base"})]

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=150
)
chunks = text_splitter.split_documents(docs)
print(f"Split external knowledge into {len(chunks)} text chunks.")

embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={'device': 'cuda'}
)

vector_store = FAISS.from_documents(chunks, embeddings_model)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("FAISS vector database built successfully.")

Split external knowledge into 2 text chunks.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


FAISS vector database built successfully.


In [38]:
augmented_prompts = []

print("Augmenting questions with retrieved context chunks...")
for idx, row in train_df.iterrows():
    question_prompt = row['prompt']
    
    retrieved_docs = retriever.invoke(question_prompt)
    context_str = " ".join([doc.page_content for doc in retrieved_docs])
    
    structured_rag_prompt = f"""Context: {context_str}
Question: {question_prompt}
Options:
A) {row['A']}
B) {row['B']}
C) {row['C']}
D) {row['D']}
E) {row['E']}
Analyze the context above and pick the top 3 most likely correct answers in ranked order."""

    augmented_prompts.append(structured_rag_prompt)

train_df['augmented_rag_prompt'] = augmented_prompts

print("\n=== Sample Augmented RAG Prompt (Row 0) ===")
print(train_df['augmented_rag_prompt'].iloc[0])

train_df.to_csv("train_augmented_rag.csv", index=False)
print("\nSaved RAG-augmented dataset to train_augmented_rag.csv")

Augmenting questions with retrieved context chunks...

=== Sample Augmented RAG Prompt (Row 0) ===
Context: Light-ion fusion is a nuclear reaction technique where light atomic nuclei combine to form heavier elements.
Tragedy of Summerhall was a catastrophic fire event in 259 AC that led to the deaths of King Aegon V.
Aegon the Conqueror forged the Iron Throne using the fiery breath of his dragon, Balerion the Black Dread.
Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings. Mean Average Precision at 3 (MAP@3) scores models based on the position of the correct answer in top 3 rankings.
Transformers rely on multi-head self-attention mechanisms to map contextual text representations dynamically.
Question: Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options.
Options:
A) Martin Heidegger believes that humans exist within a time continuum that 

In [39]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')

In [40]:
import torch 
import torch.nn as nn 
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wandb
import numpy as np 

label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
id2label = {v: k for k, v in label2id.items()}

X_text = (train_df['prompt'] + " " + train_df['A'] + " " + train_df['B'] + " " + train_df['C'] + " " + train_df['D'] + " " + train_df['E']).tolist()
X_tfidf = vectorizer.fit_transform(X_text).toarray()
y_scratch = np.array([label2id[ans] for ans in train_df['answer']])

class MCQScratchDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

scratch_dataset = MCQScratchDataset(X_tfidf, y_scratch)
scratch_loader = DataLoader(scratch_dataset, batch_size=16, shuffle=True)

wandb.init(project="smart-mcq-solver", name="Model-1-Scratch-MLP")

input_dim = X_tfidf.shape[1]
model_scratch = ScratchMCQSolver(input_dim=input_dim, hidden_dim=128).to("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

device = "cuda" if torch.cuda.is_available() else "cpu"
model_scratch.train()

for epoch in range(10):
    total_loss = 0.0
    correct = 0
    total = 0
    for batch_x, batch_y in scratch_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_scratch(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == batch_y).sum().item()
        total += batch_y.size(0)
        
    acc = correct / total
    avg_loss = total_loss / len(scratch_loader)
    wandb.log({"epoch": epoch + 1, "loss": avg_loss, "accuracy": acc})
    print(f"Epoch {epoch+1}/10 - Loss: {avg_loss:.4f} - Accuracy: {acc:.4f}")

wandb.finish()

Epoch 1/10 - Loss: 1.2028 - Accuracy: 0.7535
Epoch 2/10 - Loss: 0.1673 - Accuracy: 0.9990
Epoch 3/10 - Loss: 0.0295 - Accuracy: 1.0000
Epoch 4/10 - Loss: 0.0122 - Accuracy: 1.0000
Epoch 5/10 - Loss: 0.0066 - Accuracy: 1.0000
Epoch 6/10 - Loss: 0.0042 - Accuracy: 1.0000
Epoch 7/10 - Loss: 0.0029 - Accuracy: 1.0000
Epoch 8/10 - Loss: 0.0021 - Accuracy: 1.0000
Epoch 9/10 - Loss: 0.0016 - Accuracy: 1.0000
Epoch 10/10 - Loss: 0.0013 - Accuracy: 1.0000


accuracy,▁█████████
epoch,▁▂▃▃▄▅▆▆▇█
loss,█▂▁▁▁▁▁▁▁▁
accuracy,1
epoch,10
loss,0.00129


# RESTART

In [41]:
import os 
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
import wandb
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
from peft import LoraConfig, get_peft_model, TaskType
from dataclasses import dataclass
from typing import Optional, Union
from datasets import Dataset as HFDataset

train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv').fillna('None')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv').fillna('None')

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'\s{2,}', ' ', text)
    return text.strip()

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[col] = train_df[col].apply(clean_text)
    test_df[col] = test_df[col].apply(clean_text)

label2id = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}

In [42]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=5000)
X_train_text = (train_df['prompt'] + " " + train_df['A'] + " " + train_df['B'] + " " + train_df['C'] + " " + train_df['D'] + " " + train_df['E']).tolist()
X_test_text = (test_df['prompt'] + " " + test_df['A'] + " " + test_df['B'] + " " + test_df['C'] + " " + test_df['D'] + " " + test_df['E']).tolist()

X_train_tfidf = vectorizer.fit_transform(X_train_text).toarray()
X_test_tfidf = vectorizer.transform(X_test_text).toarray()
y_train = np.array([label2id[ans] for ans in train_df['answer']])

class MCQScratchDataset(torch.utils.data.Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

scratch_loader = DataLoader(MCQScratchDataset(X_train_tfidf, y_train), batch_size=16, shuffle=True)

class ScratchMCQSolver(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super(ScratchMCQSolver, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, 5)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

device = "cuda" if torch.cuda.is_available() else "cpu"
model_scratch = ScratchMCQSolver(input_dim=X_train_tfidf.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_scratch.parameters(), lr=1e-3)

wandb.init(project="smart-mcq-solver", name="Model-1-Scratch-MLP")
model_scratch.train()

for epoch in range(5): 
    total_loss, correct, total = 0.0, 0, 0
    for batch_x, batch_y in scratch_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model_scratch(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        correct += (torch.argmax(outputs, dim=1) == batch_y).sum().item()
        total += batch_y.size(0)
    
    wandb.log({"epoch": epoch + 1, "loss": total_loss / len(scratch_loader), "accuracy": correct / total})
wandb.finish()

accuracy,▁████
epoch,▁▃▅▆█
loss,█▂▁▁▁
accuracy,1
epoch,5
loss,0.00681


In [43]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_mcq(examples):
    first_sentences = [[prompt] * 5 for prompt in examples['prompt']]
    second_sentences = [
        [f"A: {a}", f"B: {b}", f"C: {c}", f"D: {d}", f"E: {e}"]
        for a, b, c, d, e in zip(examples['A'], examples['B'], examples['C'], examples['D'], examples['E'])
    ]
    first_sentences = sum(first_sentences, [])
    second_sentences = sum(second_sentences, [])
    
    tokenized = tokenizer(first_sentences, second_sentences, truncation=True, max_length=128) 
    return {k: [v[i:i+5] for i in range(0, len(v), 5)] for k, v in tokenized.items()}

hf_train_ds = HFDataset.from_pandas(train_df)
encoded_train_ds = hf_train_ds.map(preprocess_mcq, batched=True, remove_columns=hf_train_ds.column_names)
encoded_train_ds = encoded_train_ds.add_column("label", y_train.tolist())

hf_test_ds = HFDataset.from_pandas(test_df)
encoded_test_ds = hf_test_ds.map(preprocess_mcq, batched=True, remove_columns=hf_test_ds.column_names)

@dataclass
class DataCollatorForMultipleChoice:
    tokenizer: PreTrainedTokenizerBase
    padding: Union[bool, str, PaddingStrategy] = True
    max_length: Optional[int] = None
    pad_to_multiple_of: Optional[int] = None

    def __call__(self, features):
        label_name = "label" if "label" in features[0] else "labels"
        labels = [feature.pop(label_name) for feature in features] if label_name in features[0] else None
        batch_size = len(features)
        num_choices = len(features[0]["input_ids"])
        
        flat_features = [[{k: v[i] for k, v in feature.items()} for i in range(num_choices)] for feature in features]
        flat_features = sum(flat_features, [])
        
        batch = self.tokenizer.pad(flat_features, padding=self.padding, max_length=self.max_length, pad_to_multiple_of=self.pad_to_multiple_of, return_tensors="pt")
        batch = {k: v.view(batch_size, num_choices, -1) for k, v in batch.items()}
        if labels is not None:
            batch["labels"] = torch.tensor(labels, dtype=torch.long)
        return batch

data_collator = DataCollatorForMultipleChoice(tokenizer=tokenizer)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [44]:
model_2 = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

training_args_m2 = TrainingArguments(
    output_dir="./deberta_mcq_results",
    eval_strategy="no",
    learning_rate=1e-5,
    warmup_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8, 
    num_train_epochs=3,
    weight_decay=0.01,
    adam_epsilon=1e-6,
    logging_steps=10,
    max_grad_norm=0.5,           
    fp16=False,                  
    report_to="wandb",
    run_name="Model-2-DeBERTa-v3-Full"
)

trainer_m2 = Trainer(
    model=model_2,
    args=training_args_m2,
    train_dataset=encoded_train_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer_m2.train()
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                

Step,Training Loss
10,12.850000
20,13.012012
30,12.840527
40,12.864258
50,12.881250
60,12.985547
70,12.932617
80,12.610352
90,11.679810
100,12.320605


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▄▁▂▁▁▁▁▄█▄▂▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/learning_rate,▂▄▅▇███▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,▇█▇▇▇█▇▆▁▄▁███▇██▇██▆████████▇▇▇▇▇▇▇▇
total_flos,1137163831424880.0
train/epoch,3
train/global_step,375
train/grad_norm,10.27344
train/learning_rate,0.0
train/loss,12.86172


In [45]:
model_3_base = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query_proj", "value_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.FEATURE_EXTRACTION
)

model_3 = get_peft_model(model_3_base, lora_config)
model_3.print_trainable_parameters()

training_args_m3 = TrainingArguments(
    output_dir="./deberta_lora_results",
    learning_rate=5e-5,          
    warmup_steps=50,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    logging_steps=10,
    max_grad_norm=0.5,
    fp16=False,
    report_to="wandb",
    run_name="Model-3-DeBERTa-LoRA"
)

trainer_m3 = Trainer(
    model=model_3,
    args=training_args_m3,
    train_dataset=encoded_train_ds,
    processing_class=tokenizer,
    data_collator=data_collator,
)

trainer_m3.train()
wandb.finish()

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                

trainable params: 294,912 || all params: 184,717,825 || trainable%: 0.1597


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 2, 'bos_token_id': 1}.


Step,Training Loss
10,12.937793
20,12.941504
30,12.824512
40,12.914453
50,12.893359
60,12.915820
70,12.798242
80,12.899316
90,12.863086
100,12.972363


train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▁▁▁▃▂▁▂▃▃▄▃▄▄▄▃▄▄▇▆▄▅▅▅█▆▇▆▆▆▅▆▅█▅▆▆▆
train/learning_rate,▂▄▅▇███▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▂▂▂▂▁▁▁
train/loss,▇▇▄▆▆▆▃▆▅█▄█▆█▄▇▆▅▄▄▂▄▅▂▃▅▅▃▆▅▄▄█▄▆▁▃
total_flos,1141079389845360.0
train/epoch,3
train/global_step,375
train/grad_norm,9.29691
train/learning_rate,0.0
train/loss,12.796


In [46]:
wandb.init(project="smart-mcq-solver", name="Model-Final-Ensemble-Inference")
model_scratch.eval()
with torch.no_grad():
    logits_m1 = model_scratch(torch.tensor(X_test_tfidf, dtype=torch.float32).to(device)).cpu().numpy()

preds_m2 = trainer_m2.predict(encoded_test_ds)
logits_m2 = preds_m2.predictions

preds_m3 = trainer_m3.predict(encoded_test_ds)
logits_m3 = preds_m3.predictions

w1, w2, w3 = 0.1, 0.45, 0.45
ensemble_logits = (w1 * logits_m1) + (w2 * logits_m2) + (w3 * logits_m3)

final_predictions = []
labels = ['A', 'B', 'C', 'D', 'E']

for row_logits in ensemble_logits:
    top_3_indices = np.argsort(row_logits)[-3:][::-1]
    final_predictions.append(" ".join([labels[i] for i in top_3_indices]))

id_col = 'ID' if 'ID' in test_df.columns else 'id' if 'id' in test_df.columns else test_df.index

submission_df = pd.DataFrame({
    'ID': test_df[id_col] if id_col in test_df.columns else id_col,
    'Prediction': final_predictions
})

submission_df.to_csv('submission.csv', index=False)
print("Ensemble Saved")
wandb.finish()

Ensemble Saved


test/runtime,▁█
test/samples_per_second,█▁
test/steps_per_second,█▁
test/runtime,6.027
test/samples_per_second,82.96
test/steps_per_second,10.453
